# Eksperimen 20 — Candidate Expansion + Main-Aware Reranking (Validation Only)

Eksperimen ini hanya menggunakan **validation v2 (38 query)**.

Tujuan:
1. memperbesar candidate pool secara terkontrol;
2. melihat apakah **Candidate Hit Rate** naik;
3. melihat apakah relevant recipe tambahan benar-benar bisa dipromosikan ke Top-5 oleh reranker terbaik sementara.

Reranker yang dipakai tetap hasil terbaik Eksperimen 19:

\[
E' = 0.5E_{harmonic} + 0.5C_{main}
\]

\[
FinalScore = 0.7S_{semantic} + 0.3E'
\]

Konfigurasi retrieval yang dibandingkan:
- S50 + K20 — baseline
- S75 + K20
- S100 + K20
- S50 + K40
- S75 + K40

Ground truth hanya digunakan setelah ranking selesai untuk evaluasi.


In [1]:
import numpy as np
import pandas as pd

from dapuria_pipeline import (
    DapuriaRecipeService,
    parse_user_ingredients,
    coverage_stats,
    harmonic_efficiency,
    ingredient_matches,
    is_main_ingredient,
    add_query_prefix,
)

VAL_PATH = "ground_truth_relevance_validation_v2.csv"

SUMMARY_OUT = "hasil_candidate_expansion_mainaware_validation_ringkas.csv"
DETAIL_OUT = "hasil_candidate_expansion_mainaware_validation_detail.csv"

RETRIEVAL_CONFIGS = [
    ("S50_K20", 50, 20),
    ("S75_K20", 75, 20),
    ("S100_K20", 100, 20),
    ("S50_K40", 50, 40),
    ("S75_K40", 75, 40),
]

W_SEMANTIC = 0.7
W_EFFICIENCY = 0.3
ALPHA_HARMONIC = 0.5  # E' = .5 harmonic + .5 main coverage


def split_pipe(x):
    if pd.isna(x):
        return []
    return [s.strip() for s in str(x).split(" | ") if s.strip()]


def minmax(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) == 0:
        return arr
    lo = float(arr.min())
    hi = float(arr.max())
    return (arr - lo) / (hi - lo + 1e-9)


def hit_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    return int(any(rid in rel for rid in ranked_ids[:k]))


def precision_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    return sum(rid in rel for rid in ranked_ids[:k]) / k


def recall_at_k(ranked_ids, relevant_ids, k):
    rel = set(relevant_ids)
    if not rel:
        return 0.0
    return sum(rid in rel for rid in ranked_ids[:k]) / len(rel)


def reciprocal_rank(ranked_ids, relevant_ids):
    rel = set(relevant_ids)
    for rank, rid in enumerate(ranked_ids, start=1):
        if rid in rel:
            return 1.0 / rank
    return 0.0


def candidate_recall(candidate_ids, relevant_ids):
    rel = set(relevant_ids)
    if not rel:
        return 0.0
    return sum(rid in rel for rid in candidate_ids) / len(rel)


def main_ingredient_coverage(user_ingredients, recipe_ingredients):
    main_user = [u for u in user_ingredients if is_main_ingredient(u)]

    if main_user:
        matched = sum(
            1
            for u in main_user
            if any(ingredient_matches(u, r) for r in recipe_ingredients)
        )
        return matched / len(main_user)

    stats = coverage_stats(
        user_ingredients,
        recipe_ingredients,
    )
    return float(stats["coverage_user"])


# ============================================================
# 1. Load validation
# ============================================================
val = pd.read_csv(VAL_PATH)

required = {
    "scraped_title",
    "query_text_v2",
    "relevant_ids",
}
missing = required - set(val.columns)

if missing:
    raise ValueError(f"Kolom validation hilang: {sorted(missing)}")

dup = val.duplicated(["query_text_v2", "relevant_ids"]).sum()

if dup:
    raise ValueError(f"Validation masih punya {dup} duplicate cases.")

print("VALIDASI DATA")
print(f"Jumlah query              : {len(val)}")
print(f"Duplicate query+relevance : {dup}")


# ============================================================
# 2. Init service + corpus cache
# ============================================================
service = DapuriaRecipeService(
    chroma_dir="chroma_db",
    collection_name="resep_masakan",
)

all_docs = service._get_all_documents()

print("\nSYSTEM")
print(f"Corpus docs               : {len(all_docs)}")
print("Reranker                  : MAIN050_W07_03")
print("WLC                       : 0.7 semantic + 0.3 adjusted efficiency")
print("Adjusted efficiency       : 0.5 harmonic + 0.5 main coverage")


# ============================================================
# 3. Retrieval with configurable S and K
# ============================================================
def retrieve_configurable(
    user_text,
    user_ingredients,
    semantic_k,
    keyword_k,
):
    query_embedding = service.model.encode(
        [add_query_prefix(user_text)],
        normalize_embeddings=True,
        convert_to_numpy=True,
    )[0]

    semantic_results = service.collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=semantic_k,
        include=["documents", "metadatas", "distances"],
    )

    candidates = []

    for i, recipe_id in enumerate(semantic_results["ids"][0]):
        meta = service._parse_meta(semantic_results["metadatas"][0][i])

        candidates.append(
            {
                "id": str(recipe_id),
                **meta,
                "sbert_score": float(1 - semantic_results["distances"][0][i]),
                "retrieval_source": "semantic",
            }
        )

    existing_ids = {str(c["id"]) for c in candidates}

    keyword_scored = []

    for (
        doc_id,
        meta,
        document,
        embedding,
    ) in all_docs:
        doc_id = str(doc_id)

        if doc_id in existing_ids:
            continue

        parsed_meta = service._parse_meta(meta)
        names = parsed_meta["ingredient_names"]

        matched_count = sum(
            1 for u in user_ingredients if any(ingredient_matches(u, n) for n in names)
        )

        if matched_count <= 0:
            continue

        coverage_recipe_lexical = matched_count / len(names) if names else 0.0

        keyword_scored.append(
            (
                matched_count,
                coverage_recipe_lexical,
                doc_id,
                parsed_meta,
                embedding,
            )
        )

    keyword_scored.sort(
        key=lambda x: (
            -x[0],
            -x[1],
            str(x[2]),
        )
    )

    for (
        matched_count,
        _coverage_lexical,
        doc_id,
        parsed_meta,
        embedding,
    ) in keyword_scored[:keyword_k]:
        semantic_score = float(
            np.dot(
                query_embedding,
                np.asarray(embedding, dtype=float),
            )
        )

        candidates.append(
            {
                "id": doc_id,
                **parsed_meta,
                "sbert_score": semantic_score,
                "retrieval_source": "keyword",
            }
        )

    return candidates


# ============================================================
# 4. Rerank with frozen MAIN050_W07_03
# ============================================================
def rerank_mainaware(
    candidates,
    user_ingredients,
):
    sem_mm = minmax([float(c["sbert_score"]) for c in candidates])

    ranked = []

    for c, s_norm in zip(
        candidates,
        sem_mm,
    ):
        stats = coverage_stats(
            user_ingredients,
            c["ingredient_names"],
        )

        harmonic = harmonic_efficiency(
            stats["coverage_recipe"],
            stats["coverage_user"],
        )

        main_cov = main_ingredient_coverage(
            user_ingredients,
            c["ingredient_names"],
        )

        adjusted_eff = ALPHA_HARMONIC * harmonic + (1 - ALPHA_HARMONIC) * main_cov

        final_score = W_SEMANTIC * float(s_norm) + W_EFFICIENCY * float(adjusted_eff)

        ranked.append(
            {
                **c,
                "semantic_minmax": float(s_norm),
                "harmonic_efficiency": float(harmonic),
                "main_coverage": float(main_cov),
                "adjusted_efficiency": float(adjusted_eff),
                "final_score": float(final_score),
            }
        )

    ranked.sort(
        key=lambda x: (
            -x["final_score"],
            -x["semantic_minmax"],
            str(x["id"]),
        )
    )

    return ranked


# ============================================================
# 5. Evaluate all retrieval configurations
# ============================================================
summary_rows = []
detail_rows = []

for config_name, semantic_k, keyword_k in RETRIEVAL_CONFIGS:
    print(
        f"\nMenjalankan {config_name} (semantic={semantic_k}, keyword={keyword_k})..."
    )

    candidate_hits = []
    candidate_recalls = []

    h5s = []
    h10s = []

    p5s = []
    p10s = []

    r5s = []
    r10s = []

    rrs = []
    pool_sizes = []

    retrieval_failures = 0
    ranking_failures_top5 = 0
    ranking_failures_top10 = 0

    for _, row in val.iterrows():
        query_text = str(row["query_text_v2"]).strip()

        relevant_ids = set(split_pipe(row["relevant_ids"]))

        user_ingredients = parse_user_ingredients(query_text)

        candidates = retrieve_configurable(
            query_text,
            user_ingredients,
            semantic_k=semantic_k,
            keyword_k=keyword_k,
        )

        candidate_ids = [str(c["id"]) for c in candidates]

        cand_hit = int(any(rid in relevant_ids for rid in candidate_ids))

        cand_recall = candidate_recall(
            candidate_ids,
            relevant_ids,
        )

        ranked = rerank_mainaware(
            candidates,
            user_ingredients,
        )

        ranked_ids = [str(c["id"]) for c in ranked]

        h5 = hit_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        h10 = hit_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )

        p5 = precision_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        p10 = precision_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )

        r5 = recall_at_k(
            ranked_ids,
            relevant_ids,
            5,
        )
        r10 = recall_at_k(
            ranked_ids,
            relevant_ids,
            10,
        )

        rr = reciprocal_rank(
            ranked_ids,
            relevant_ids,
        )

        if not cand_hit:
            failure_type = "retrieval_failure"
            retrieval_failures += 1
        elif not h10:
            failure_type = "ranking_failure_outside_top10"
            ranking_failures_top10 += 1
            ranking_failures_top5 += 1
        elif not h5:
            failure_type = "ranking_failure_rank6_10"
            ranking_failures_top5 += 1
        else:
            failure_type = "success_top5"

        best_rank = None
        best_title = None

        for rank, c in enumerate(
            ranked,
            start=1,
        ):
            if str(c["id"]) in relevant_ids:
                best_rank = rank
                best_title = c["title"]
                break

        relevant_in_pool = sum(rid in relevant_ids for rid in candidate_ids)

        candidate_hits.append(cand_hit)
        candidate_recalls.append(cand_recall)

        h5s.append(h5)
        h10s.append(h10)

        p5s.append(p5)
        p10s.append(p10)

        r5s.append(r5)
        r10s.append(r10)

        rrs.append(rr)
        pool_sizes.append(len(candidate_ids))

        detail_rows.append(
            {
                "config": config_name,
                "semantic_k": semantic_k,
                "keyword_k": keyword_k,
                "scraped_title": row["scraped_title"],
                "query_text": query_text,
                "n_relevant_ids": len(relevant_ids),
                "relevant_ids": " | ".join(sorted(relevant_ids)),
                "candidate_pool_size": len(candidate_ids),
                "relevant_in_candidate_pool": relevant_in_pool,
                "candidate_hit": cand_hit,
                "candidate_recall": cand_recall,
                "hit_at_5": h5,
                "hit_at_10": h10,
                "precision_at_5": p5,
                "precision_at_10": p10,
                "recall_at_5": r5,
                "recall_at_10": r10,
                "reciprocal_rank": rr,
                "best_relevant_rank": best_rank,
                "best_relevant_title": best_title,
                "failure_type": failure_type,
                "top10_result_ids": " | ".join(ranked_ids[:10]),
                "top10_result_titles": " | ".join(str(c["title"]) for c in ranked[:10]),
            }
        )

    summary_rows.append(
        {
            "config": config_name,
            "semantic_k": semantic_k,
            "keyword_k": keyword_k,
            "n_query": len(val),
            "avg_candidate_pool_size": float(np.mean(pool_sizes)),
            "candidate_hit_rate": float(np.mean(candidate_hits)),
            "candidate_hit_count": int(np.sum(candidate_hits)),
            "candidate_recall_macro": float(np.mean(candidate_recalls)),
            "hit_at_5": float(np.mean(h5s)),
            "hit_at_5_count": int(np.sum(h5s)),
            "hit_at_10": float(np.mean(h10s)),
            "hit_at_10_count": int(np.sum(h10s)),
            "precision_at_5_macro": float(np.mean(p5s)),
            "precision_at_10_macro": float(np.mean(p10s)),
            "recall_at_5_macro": float(np.mean(r5s)),
            "recall_at_10_macro": float(np.mean(r10s)),
            "mrr": float(np.mean(rrs)),
            "retrieval_failure_count": int(retrieval_failures),
            "ranking_failure_top5_count": int(ranking_failures_top5),
            "ranking_failure_top10_count": int(ranking_failures_top10),
        }
    )


# ============================================================
# 6. Save and print
# ============================================================
summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        [
            "hit_at_5",
            "candidate_hit_rate",
            "hit_at_10",
            "mrr",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

detail_df = pd.DataFrame(detail_rows)

summary_df.to_csv(
    SUMMARY_OUT,
    index=False,
    encoding="utf-8-sig",
)

detail_df.to_csv(
    DETAIL_OUT,
    index=False,
    encoding="utf-8-sig",
)

print("\n" + "=" * 125)
print("HASIL CANDIDATE EXPANSION + MAIN-AWARE RERANKING - VALIDATION ONLY")
print("=" * 125)

print(summary_df.to_string(index=False))

best = summary_df.iloc[0]

baseline = summary_df[summary_df["config"] == "S50_K20"].iloc[0]

print("\nKONFIGURASI TERBAIK VALIDATION:")
print(
    f"{best['config']} | "
    f"CandidateHit={best['candidate_hit_rate']:.4f} "
    f"({int(best['candidate_hit_count'])}/{int(best['n_query'])}) | "
    f"CandidateRecall={best['candidate_recall_macro']:.4f} | "
    f"Hit@5={best['hit_at_5']:.4f} "
    f"({int(best['hit_at_5_count'])}/{int(best['n_query'])}) | "
    f"Hit@10={best['hit_at_10']:.4f} | "
    f"MRR={best['mrr']:.4f}"
)

print("\nBASELINE RETRIEVAL + MAIN-AWARE RERANKER:")
print(
    f"S50_K20 | "
    f"CandidateHit={baseline['candidate_hit_rate']:.4f} | "
    f"CandidateRecall={baseline['candidate_recall_macro']:.4f} | "
    f"Hit@5={baseline['hit_at_5']:.4f} | "
    f"Hit@10={baseline['hit_at_10']:.4f} | "
    f"MRR={baseline['mrr']:.4f}"
)

print("\nDIAGNOSTIK KONFIGURASI TERBAIK:")
print(
    f"Retrieval failure          : "
    f"{int(best['retrieval_failure_count'])}/{int(best['n_query'])}"
)
print(
    f"Candidate-hit tapi >Top5   : "
    f"{int(best['ranking_failure_top5_count'])}/{int(best['n_query'])}"
)
print(
    f"Candidate-hit tapi >Top10  : "
    f"{int(best['ranking_failure_top10_count'])}/{int(best['n_query'])}"
)

print("\nPENTING:")
print("- Pemilihan konfigurasi hanya memakai validation v2.")
print("- 90-query test lama tidak digunakan untuk tuning.")
print("- Semua konfigurasi memakai reranker MAIN050_W07_03 yang sama.")
print(
    "- Tujuan eksperimen ini adalah menguji candidate expansion, bukan mengubah formula reranking lagi."
)

print("\nOUTPUT:")
print(f"- {SUMMARY_OUT}")
print(f"- {DETAIL_OUT}")

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VALIDASI DATA
Jumlah query              : 38
Duplicate query+relevance : 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4915.33it/s]



SYSTEM
Corpus docs               : 5000
Reranker                  : MAIN050_W07_03
WLC                       : 0.7 semantic + 0.3 adjusted efficiency
Adjusted efficiency       : 0.5 harmonic + 0.5 main coverage

Menjalankan S50_K20 (semantic=50, keyword=20)...

Menjalankan S75_K20 (semantic=75, keyword=20)...

Menjalankan S100_K20 (semantic=100, keyword=20)...

Menjalankan S50_K40 (semantic=50, keyword=40)...

Menjalankan S75_K40 (semantic=75, keyword=40)...

HASIL CANDIDATE EXPANSION + MAIN-AWARE RERANKING - VALIDATION ONLY
  config  semantic_k  keyword_k  n_query  avg_candidate_pool_size  candidate_hit_rate  candidate_hit_count  candidate_recall_macro  hit_at_5  hit_at_5_count  hit_at_10  hit_at_10_count  precision_at_5_macro  precision_at_10_macro  recall_at_5_macro  recall_at_10_macro      mrr  retrieval_failure_count  ranking_failure_top5_count  ranking_failure_top10_count
 S75_K40          75         40       38                    115.0            0.763158                   29  